# Simple Feature Engineering: Project Walkthrough

This notebook is a **hybrid guide**: it explains the project architecture and also runs a small toy example end to end.

The project builds features from stock OHLCV data.

- **O**pen: first traded price in a bar
- **H**igh: maximum traded price in a bar
- **L**ow: minimum traded price in a bar
- **C**lose: last traded price in a bar
- **V**olume: shares traded in a bar


## 1) Project in one minute

The core workflow is deliberately small:

```text
load data -> clean invalid rows -> compute features -> export outputs
```

Where each stage lives:

- `src/feature_engineering/pipeline/load.py`: reads OHLCV from CSV or ClickHouse
- `src/feature_engineering/pipeline/clean.py`: removes rows that violate market-data invariants
- `src/feature_engineering/pipeline/engineer.py`: computes configured feature columns per symbol
- `src/feature_engineering/pipeline/export.py`: writes Parquet/CSV outputs plus run metadata
- `src/feature_engineering/features/*.py`: feature formulas grouped by category
- `src/feature_engineering/features/registry.py`: registry mapping function name -> feature metadata


In [1]:
from pathlib import Path
import sys

import pandas as pd


def resolve_project_root() -> Path:
    """Resolve project root when notebook runs from root or notebooks/."""
    current = Path.cwd().resolve()

    if (current / "src").exists():
        return current

    if current.name == "notebooks" and (current.parent / "src").exists():
        return current.parent

    raise RuntimeError("Could not find project root containing src/.")


PROJECT_ROOT = resolve_project_root()
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from feature_engineering.features.registry import REGISTRY  # noqa: E402
from feature_engineering.pipeline.clean import clean_ohlcv  # noqa: E402
from feature_engineering.pipeline.engineer import compute_features  # noqa: E402

print(f"Project root: {PROJECT_ROOT}")
print(f"Loaded {len(REGISTRY)} registered feature functions.")

Project root: /home/ai4000/projects/feature-engineering
Loaded 11 registered feature functions.


## 2) Current source tree (sanity check)

This confirms the codebase shape that this notebook describes.


In [2]:
def print_tree(root: Path, max_depth: int = 2) -> None:
    """Print a compact directory tree up to max_depth."""
    root = root.resolve()

    for path in sorted(root.rglob("*")):
        rel = path.relative_to(root)
        depth = len(rel.parts)

        if depth > max_depth:
            continue

        indent = "  " * (depth - 1)
        suffix = "/" if path.is_dir() else ""
        print(f"{indent}{rel}{suffix}")


print("Top of src tree:")
print_tree(PROJECT_ROOT / "src", max_depth=2)

Top of src tree:
GUIDE_src.md
feature_engineering.egg-info/
  feature_engineering.egg-info/PKG-INFO
  feature_engineering.egg-info/SOURCES.txt
  feature_engineering.egg-info/dependency_links.txt
  feature_engineering.egg-info/entry_points.txt
  feature_engineering.egg-info/requires.txt
  feature_engineering.egg-info/top_level.txt
features/
  features/GUIDE_features.md
  features/__init__.py
  features/__pycache__/
  features/registry.py
  features/returns.py
  features/trend.py
  features/volatility.py
  features/volume.py
main.py
pipeline/
  pipeline/GUIDE_pipeline.md
  pipeline/__init__.py
  pipeline/__pycache__/
  pipeline/clean.py
  pipeline/cli.py
  pipeline/engineer.py
  pipeline/export.py
  pipeline/load.py
run.py


## 3) Feature menu and categories

Every feature is registered with metadata:

- `category`: group used by include/exclude filters
- `description`: plain-language meaning
- `calculation`: compact formula summary

Categories in this project are `returns`, `trend`, `volatility`, `volume`, and `target`.

`target` features are forward-looking labels and should usually be excluded from live model inputs.


In [3]:
feature_rows = []

for function_name, spec in sorted(REGISTRY.items()):
    feature_rows.append(
        {
            "function": function_name,
            "category": spec.category,
            "description": spec.description,
            "calculation": spec.calculation,
        }
    )

feature_menu = pd.DataFrame(feature_rows)
feature_menu

,function,category,description,calculation
0,bar_range_pct,volatility,High-low bar range as a fraction of close.,(high_t - low_t) / close_t
1,dollar_volume,volume,Dollar value traded in each bar.,close_t * volume_t
2,log_return,returns,Natural-log return from one close to the next.,ln(close_t / close_{t-1})
3,moving_average,trend,Simple moving average of close prices over a r...,mean(close) over trailing window
4,next_n_day_return,target,Forward N-trading-day return target based on e...,EOD_close_{day+n} / EOD_close_day - 1
5,price_vs_sma,trend,Close price divided by its moving average minu...,close_t / moving_average_t - 1
6,rate_of_change,trend,Rate of change over a fixed number of rows.,close_t / close_{t-periods} - 1
7,rolling_std,volatility,Rolling standard deviation of log returns.,std(ln(close_t / close_{t-1})) over trailing w...
8,simple_return,returns,Simple percentage return from one close to the...,close_t / close_{t-1} - 1
9,volume_change,volume,One-period percentage change in volume.,volume_t / volume_{t-1} - 1


## 4) Config mental model

The pipeline behavior is driven by `config.toml`:

- `[run]`: source, symbols, date range, output format
- `[data_quality]`: row-drop rules for invalid bars
- `[features]`: category filters plus `[[features.params]]` feature list

Each `[[features.params]]` entry has:

- `name`: output column name
- `fn`: registry function key
- function-specific parameters (for example `window` or `days`)
- `enabled`: whether to compute this column


## 5) Mini end-to-end demo on toy data

This demo runs only in memory (no database and no file writes):

1. Build a tiny OHLCV table for two symbols
2. Clean invalid bars
3. Compute selected features
4. Inspect results and warmup `NaN` behavior


In [4]:
toy_rows = [
    {
        "symbol": "AAPL",
        "ts": "2024-01-02 09:30:00",
        "open": 100.0,
        "high": 101.0,
        "low": 99.5,
        "close": 100.5,
        "volume": 1000,
    },
    {
        "symbol": "AAPL",
        "ts": "2024-01-03 09:30:00",
        "open": 100.5,
        "high": 102.0,
        "low": 100.0,
        "close": 101.8,
        "volume": 1300,
    },
    {
        "symbol": "AAPL",
        "ts": "2024-01-04 09:30:00",
        "open": 101.8,
        "high": 103.0,
        "low": 101.2,
        "close": 102.5,
        "volume": 900,
    },
    {
        "symbol": "AAPL",
        "ts": "2024-01-05 09:30:00",
        "open": 102.5,
        "high": 102.2,
        "low": 102.3,
        "close": 102.4,
        "volume": 1100,
    },
    {
        "symbol": "MSFT",
        "ts": "2024-01-02 09:30:00",
        "open": 250.0,
        "high": 251.0,
        "low": 249.0,
        "close": 250.8,
        "volume": 2000,
    },
    {
        "symbol": "MSFT",
        "ts": "2024-01-03 09:30:00",
        "open": 250.8,
        "high": 252.2,
        "low": 250.0,
        "close": 251.7,
        "volume": 2200,
    },
    {
        "symbol": "MSFT",
        "ts": "2024-01-04 09:30:00",
        "open": 251.7,
        "high": 253.0,
        "low": 251.2,
        "close": 252.5,
        "volume": 2400,
    },
    {
        "symbol": "MSFT",
        "ts": "2024-01-05 09:30:00",
        "open": 260.0,
        "high": 253.2,
        "low": 252.0,
        "close": 252.7,
        "volume": 2600,
    },
]

toy_raw = pd.DataFrame(toy_rows)
toy_raw["ts"] = pd.to_datetime(toy_raw["ts"])

print("Raw toy data:")
toy_raw

Raw toy data:


,symbol,ts,open,high,low,close,volume
0,AAPL,2024-01-02 09:30:00,100.0,101.0,99.5,100.5,1000
1,AAPL,2024-01-03 09:30:00,100.5,102.0,100.0,101.8,1300
2,AAPL,2024-01-04 09:30:00,101.8,103.0,101.2,102.5,900
3,AAPL,2024-01-05 09:30:00,102.5,102.2,102.3,102.4,1100
4,MSFT,2024-01-02 09:30:00,250.0,251.0,249.0,250.8,2000
5,MSFT,2024-01-03 09:30:00,250.8,252.2,250.0,251.7,2200
6,MSFT,2024-01-04 09:30:00,251.7,253.0,251.2,252.5,2400
7,MSFT,2024-01-05 09:30:00,260.0,253.2,252.0,252.7,2600


In [5]:
data_quality_config = {
    "drop_zero_prices": True,
    "drop_high_lt_low": True,
    "drop_ohlc_violations": True,
}

toy_clean, cleaning_report = clean_ohlcv(toy_raw, data_quality=data_quality_config)

print("Cleaning report:")
print(cleaning_report)

print("\nCleaned data:")
toy_clean

Cleaning report:
{'initial_rows': 8, 'rules': {'drop_zero_prices': {'enabled': True, 'dropped': 0, 'reason': 'open, high, low, and close must be positive prices'}, 'drop_high_lt_low': {'enabled': True, 'dropped': 1, 'reason': 'high must be greater than or equal to low'}, 'drop_ohlc_violations': {'enabled': True, 'dropped': 1, 'reason': 'open and close must sit inside the low-high range'}}, 'final_rows': 6, 'total_dropped': 2}

Cleaned data:


,symbol,ts,open,high,low,close,volume
0,AAPL,2024-01-02 09:30:00,100.0,101.0,99.5,100.5,1000
1,AAPL,2024-01-03 09:30:00,100.5,102.0,100.0,101.8,1300
2,AAPL,2024-01-04 09:30:00,101.8,103.0,101.2,102.5,900
3,MSFT,2024-01-02 09:30:00,250.0,251.0,249.0,250.8,2000
4,MSFT,2024-01-03 09:30:00,250.8,252.2,250.0,251.7,2200
5,MSFT,2024-01-04 09:30:00,251.7,253.0,251.2,252.5,2400


In [6]:
demo_config = {
    "features": {
        "include_categories": [],
        "exclude_categories": [],
        "params": [
            {"name": "log_return", "fn": "log_return", "enabled": True},
            {"name": "ma_3", "fn": "moving_average", "window": 3, "enabled": True},
            {
                "name": "price_vs_sma_3",
                "fn": "price_vs_sma",
                "window": 3,
                "enabled": True,
            },
            {
                "name": "rolling_std_3",
                "fn": "rolling_std",
                "window": 3,
                "enabled": True,
            },
            {
                "name": "volume_ratio_3",
                "fn": "volume_ratio",
                "window": 3,
                "enabled": True,
            },
            {
                "name": "next_1d_return",
                "fn": "next_n_day_return",
                "days": 1,
                "enabled": True,
            },
        ],
    }
}

toy_features = compute_features(toy_clean, demo_config)

print("Feature frame (identifier columns + engineered features):")
toy_features

Feature frame (identifier columns + engineered features):


,symbol,ts,log_return,ma_3,price_vs_sma_3,rolling_std_3,volume_ratio_3,next_1d_return
0,AAPL,2024-01-02 09:30:00,NaN,NaN,NaN,NaN,NaN,0.012935
1,AAPL,2024-01-03 09:30:00,0.012852,NaN,NaN,NaN,NaN,0.006876
2,AAPL,2024-01-04 09:30:00,0.006853,101.600000,0.008858,0.004242,0.843750,NaN
3,MSFT,2024-01-02 09:30:00,NaN,NaN,NaN,NaN,NaN,0.003589
4,MSFT,2024-01-03 09:30:00,0.003582,NaN,NaN,NaN,NaN,0.003178
5,MSFT,2024-01-04 09:30:00,0.003173,251.666667,0.003311,0.000289,1.090909,NaN


In [7]:
print("Missing-value fraction by feature column:")
toy_features.isna().mean().sort_values(ascending=False)

Missing-value fraction by feature column:


ma_3              0.666667
volume_ratio_3    0.666667
rolling_std_3     0.666667
price_vs_sma_3    0.666667
log_return        0.333333
next_1d_return    0.333333
symbol            0.000000
ts                0.000000
dtype: float64

### Interpreting the demo output

- Early rows have `NaN` values for rolling-window features because there is not enough history yet.
- `next_1d_return` is a **target label** (future information), so keep it out of live features.
- Computation is done independently per `symbol`, which avoids cross-symbol leakage.


## 6) Where to make changes

Use this routing guide when you work on the project:

- Add or change feature formulas: `src/feature_engineering/features/returns.py`, `src/feature_engineering/features/trend.py`, `src/feature_engineering/features/volatility.py`, `src/feature_engineering/features/volume.py`
- Register feature metadata: `src/feature_engineering/features/registry.py`
- Adjust data-quality rules: `src/feature_engineering/pipeline/clean.py`
- Adjust feature orchestration and category filtering: `src/feature_engineering/pipeline/engineer.py`
- Adjust loading behavior (CSV or ClickHouse): `src/feature_engineering/pipeline/load.py`
- Adjust outputs and summary artifacts: `src/feature_engineering/pipeline/export.py`
- Update test coverage: `tests/test_feature_math.py`, `tests/test_simple_pipeline.py`


## 7) Common failure modes and checks

- **Lookahead bias**: never feed `target` columns into live prediction inputs.
- **Window alignment errors**: verify rolling features use only current and past rows.
- **Silent `NaN` spread**: check missing-value rates before training.
- **Cross-symbol leakage**: keep grouped-by-symbol computation for every feature.
- **Simple vs. log return mismatch**: be explicit about which return convention you use downstream.


## 8) Useful commands

```bash
uv run python run.py --config config.toml
uv run feature-pipeline --config config.toml
uv run pytest -q
```

Tip: when exploring quickly, set `source = "csv"` in `config.toml` and point `input_path` to a small local file.
